In [ ]:
import os, sys, math, ssl, io, pytz, numpy as np, pandas as pd, requests
from datetime import datetime, timedelta, date
from timezonefinder import TimezoneFinder
from meteostat import Stations, Hourly
from isd import Batch
from scp import SCPClient
import paramiko
import calendar
from pandas.errors import EmptyDataError

from methods import *



# Define constants
year = 2024
file_type = 'AMY'
save_folder = f'epws_wmo_{year}'

# Check if the 'zipcodes' variable is already defined
if 'zipcodes' not in globals():
    # Load the zip codes CSV only if 'zipcodes' is not already defined
    zipcodes = pd.read_csv(f'resources/zip_code_list_{year}.csv', dtype={f'EPW_file_name_{year}': str, f'weather_station_wmo_{year}': str})

# Initialize a counter for iterations
counter = 0

# Process each row in the DataFrame starting from the specified index
for index, row in zipcodes.iloc[0:].iterrows():
    # if float(row.get(f"distance_location_station_miles_{year}")) < 50:
    #     continue
    print(index)


    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed
    print(zip_code)
    lat = row['lat']
    lon = row['lng']
    save_name = None

    # print(lat)
    # print(lon)
    # print(row['city'])

    # Retrieve data for the current location
    # retrieve_status, distance, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    retrieve_status, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations, flags = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    if retrieve_info_closest_other_locations:
        try:
            # retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
            retrieve_status, hdd, cdd, flags= retrieve_info_other_location(wmo, zipcodes, year)
            # print(retrieve_status)
            # print('=====')
            # print(wmo)
        except IndexError:
            # retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
            print('**********')
            print(wmo)
            retrieve_status, hdd, cdd, flags = retrieve_info_other_location(wmo, zipcodes, year)





    # Validate that retrieve_status is a boolean
    if not isinstance(bool(retrieve_status), bool):
        raise TypeError(f"retrieve_status is not a boolean. Actual value: {retrieve_status}. Program stopped.")
    
    [distance_mi, lat_station, lon_station] = retrieve_distance_station_location(wmo, lat, lon)

    # Update the DataFrame only if the cell is empty or contains a placeholder (like 'nan')
    update_if_missing(zipcodes, index, f"EPW_file_name_{year}", f"{wmo}_{year}.epw")
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance_mi)  # Convert from meters to miles
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)
    update_if_missing(zipcodes, index, f"Tdb_holes_{year}", flags[6])
    update_if_missing(zipcodes, index, f"Tdew_holes_{year}", flags[7])
    update_if_missing(zipcodes, index, f"RH_holes_{year}", flags[8])

    # Increment the counter
    counter += 1

    # Every 10 iterations, save the DataFrame and reopen it
    if counter % 10 == 0:
        # Save the DataFrame to the CSV file
        zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)

        # Reopen the file to ensure the latest version is loaded
        # zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# After the loop is done, ensure the latest state is saved
zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)




## Test that the results are fine

In [1]:
import os
import shutil
import subprocess
import openstudio
import pandas as pd

def check_epw_quality(epw_path):
    """
    Perform quality checks on an EPW file and return a summary of whether each variable is 'Good' or 'Suspicious'.
    
    Parameters:
        epw_path (str): Path to the EPW file.
    
    Returns:
        dict: Dictionary with quality check results for each tested variable.
    """
    # Read EPW file (skip header, start from line 9)
    epw_df = pd.read_csv(epw_path, skiprows=8, header=None)

    # Dictionary to store check results
    quality_checks = {}

    ### 1️⃣ Check for Missing Data ###
    missing_values = epw_df.isnull().sum().sum()
    quality_checks["Missing Data"] = "Suspicious" if missing_values > 0 else "Good"

    ### 2️⃣ Enhanced Dry Bulb Temperature Checks ###
    dry_bulb_temp = epw_df[6]
    month = epw_df[1]

    # Extreme values (-50°C to 60°C)
    extreme_values = ((dry_bulb_temp < -50) | (dry_bulb_temp > 60)).any()
    quality_checks["Extreme Temperature"] = "Suspicious" if extreme_values else "Good"

    # Sudden jumps (> 15°C per hour)
    temp_diff = dry_bulb_temp.diff().abs()
    rapid_jumps = (temp_diff > 15).any()
    quality_checks["Sudden Temp Jumps"] = "Suspicious" if rapid_jumps else "Good"

    # Constant temperature for more than 12 hours
    const_periods = (dry_bulb_temp.rolling(window=12, min_periods=1).std() == 0).any()
    quality_checks["Constant Temp Periods"] = "Suspicious" if const_periods else "Good"

    # Unrealistic day-night swings (<3°C or >30°C)
    epw_df["daily_max"] = dry_bulb_temp.groupby(epw_df[2]).transform("max")
    epw_df["daily_min"] = dry_bulb_temp.groupby(epw_df[2]).transform("min")
    epw_df["daily_range"] = epw_df["daily_max"] - epw_df["daily_min"]
    unrealistic_swings = ((epw_df["daily_range"] < 3) | (epw_df["daily_range"] > 30)).any()
    quality_checks["Day-Night Swings"] = "Suspicious" if unrealistic_swings else "Good"

    # Seasonal temperature mismatches
    summer_issues = ((month.isin([6, 7, 8])) & (dry_bulb_temp < 0)).any()  # Summer months with freezing temps
    winter_issues = ((month.isin([12, 1, 2])) & (dry_bulb_temp > 40)).any()  # Winter months with extreme heat
    quality_checks["Summer Freezing"] = "Suspicious" if summer_issues else "Good"
    quality_checks["Winter Extreme Heat"] = "Suspicious" if winter_issues else "Good"

    # Missing temperature data
    missing_temps = dry_bulb_temp.isna().any()
    quality_checks["Missing Temperature Data"] = "Suspicious" if missing_temps else "Good"

    return quality_checks

def run_energyplus_simulations(year):
    input_csv = f'resources/zip_code_list_{year}.csv'
    # Load the input CSV.
    df = pd.read_csv(input_csv)
    
    # Add the "EnergyPlus Status" column if it doesn't exist.
    if "EnergyPlus Status" not in df.columns:
        df["EnergyPlus Status"] = ""
    
    # Ensure quality check columns exist.
    quality_columns = [
        "Missing Data",
        "Extreme Temperature",
        "Sudden Temp Jumps",
        "Constant Temp Periods",
        "Day-Night Swings",
        "Summer Freezing",
        "Winter Extreme Heat",
        "Missing Temperature Data"
    ]
    for col in quality_columns:
        if col not in df.columns:
            df[col] = ""
    
    # Get the unique weather station IDs.
    unique_stations = df[f"weather_station_wmo_{year}"].unique()

    # Set EnergyPlus installation location.
    energyplus_path = '/Applications/EnergyPlus-23-2-0/energyplus'
    current_directory = os.getcwd()


    print('***************')
    print(len(unique_stations))
    print('***************')

    k = 0
    for station in unique_stations:
        print(k)

        # If any row with this weather station already has a non-empty EnergyPlus Status, skip simulation.
        station_rows = df[df[f"weather_station_wmo_{year}"] == station]
        if station_rows["EnergyPlus Status"].notna().all() and (station_rows["EnergyPlus Status"] != "").all():
            k += 1
            continue

        epw_path = os.path.join(f"epws_wmo_{year}", f'{station}_{year}.epw')

        # Create (or reuse) the simulation directory.
        out_dir = "test_epws_dir"
        os.makedirs(out_dir, exist_ok=True)

        min_idf = openstudio.IdfFile().load("resources/min.idf").get()
        # Save IDF file
        min_idf.save(f"{out_dir}/in.idf", True)

        # Copy the minimal IDF file and the EPW file into the simulation directory.
        shutil.copy(epw_path, os.path.join(out_dir, "in.epw"))

        # Change directory to the simulation folder.
        os.chdir(out_dir)

        # Run EnergyPlus simulation.
        process = subprocess.run(
            [energyplus_path, "-w", "in.epw", "in.idf"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        
        # Check if the simulation ran successfully.
        status = "Good" if process.stderr.strip() == "EnergyPlus Completed Successfully." else "Bad"

        # Return to the original directory.
        os.chdir(current_directory)

        # Update simulation status for all rows corresponding to this weather station.
        df.loc[df[f"weather_station_wmo_{year}"] == station, "EnergyPlus Status"] = status

        # Run quality checks on the EPW file.
        quality_results = check_epw_quality(epw_path)
        for key, value in quality_results.items():
            df.loc[df[f"weather_station_wmo_{year}"] == station, key] = value

        # Save and reload CSV after processing each station.
        df.to_csv(input_csv, index=False)
        df = pd.read_csv(input_csv)

        k += 1

    return df

year = 2024
# Run the EnergyPlus simulations, perform EPW quality checks, and print the updated DataFrame.
final_df = run_energyplus_simulations(year)
final_df
